# 从 Materials Project 下载含 Na 且 band_gap ≥ 1 eV 的材料

**目标**：筛选同时满足以下条件的材料，下载 CIF 结构文件及其主要性质信息。

- 含 Na（`elements` 包含 `Na`）
- 带隙 `band_gap ≥ 1.0 eV`

**产出**（位于 `cif/na_bandgap_ge1/`）：

| 文件 | 内容 |
|------|------|
| `summary.csv` / `summary.json` | 每个材料的关键性质（formula、band_gap、密度、对称性等） |
| `*.cif` | 各材料的晶体结构 CIF 文件 |
| `download_report.csv` | 每个 material_id 的下载状态（ok / skipped / error） |
| `summary_with_status.csv` | 性质表 + 下载状态合并的最终清单 |

> 依赖：`pip install mp-api pymatgen pandas`

## 1. 查询 Materials Project

用 `mpr.materials.summary.search` 做服务端筛选：

- `elements=["Na"]` —— 材料须含 Na
- `band_gap=(1.0, None)` —— 带隙 ≥ 1 eV（上界不限）

一次性取回结构与关键性质字段。
## 2. 构建信息表并预览

把 `docs` 整理成长表，保存为 `summary.csv` / `summary.json`。

In [1]:
# ---- Python 3.10 兼容性 shim（必须在导入 mp_api 之前）----
# emmet-core 新版在 emmet/core/tasks.py 里直接 `from typing import NotRequired`，
# 但 NotRequired/Required 是 Python 3.11+ 才进标准库 typing 的。
# 这里先从 typing_extensions 把它们补丁进 typing，使 Python 3.10 也能正常导入。
import typing
import typing_extensions

for _name in ("NotRequired", "Required"):
    if not hasattr(typing, _name) and hasattr(typing_extensions, _name):
        setattr(typing, _name, getattr(typing_extensions, _name))

import os
from pathlib import Path

import pandas as pd
from mp_api.client import MPRester
from pymatgen.io.cif import CifWriter


# -------- 配置 --------
# API key 解析顺序: 环境变量 MP_API_KEY -> ./.env -> demo.py 默认 key
def _resolve_api_key() -> str:
    key = os.environ.get("MP_API_KEY", "").strip()
    if key:
        return key
    dotenv = Path(".env")
    if dotenv.is_file():
        for raw in dotenv.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if line.startswith("export "):
                line = line[len("export "):].strip()
            if "=" not in line or line.startswith("#"):
                continue
            k, v = line.split("=", 1)
            if k.strip() == "MP_API_KEY":
                return v.strip().strip('"').strip("'")
    return "fFtrdShVJH4jwWHiId8v4cyGzV2oYnoG"  # 回退到 demo.py 默认 key


API_KEY = _resolve_api_key()

# 筛选条件
NA_ELEMENT = "Na"
BAND_GAP_MIN = 1.0          # band_gap >= 1 eV

# 输出
OUTDIR = Path("cif/na_bandgap_ge1")
OUTDIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = OUTDIR / "summary.csv"
SUMMARY_JSON = OUTDIR / "summary.json"
REPORT_CSV = OUTDIR / "download_report.csv"
FINAL_CSV = OUTDIR / "summary_with_status.csv"

# 下载选项
LIMIT = 0          # >0 时只处理前 N 条（测试用）；0 = 全部
SKIP_EXISTING = True  # 已存在的 CIF 跳过

print(f"Python {'.'.join(map(str, __import__('sys').version_info[:3]))}  "
      f"NotRequired 可用: {hasattr(typing, 'NotRequired')}")
print(f"输出目录: {OUTDIR.resolve()}")
print(f"筛选条件: 含 {NA_ELEMENT} 且 band_gap >= {BAND_GAP_MIN} eV")
print(f"LIMIT={LIMIT}  SKIP_EXISTING={SKIP_EXISTING}")

/Users/north./anaconda3/envs/north/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.10.19  NotRequired 可用: True
输出目录: /Users/north./Documents/JXNU/code/self_study/21-mp-download/cif/na_bandgap_ge1
筛选条件: 含 Na 且 band_gap >= 1.0 eV
LIMIT=0  SKIP_EXISTING=True


In [2]:
fields = [
    "material_id", "formula_pretty", "formula_anonymous",
    "band_gap", "energy_above_hull", "density", "volume",
    "nsites", "nelements", "elements", "symmetry",
    "structure",  # 用于写 CIF
]

with MPRester(API_KEY) as mpr:
    # 这里去掉 elements和band_gap就可以不做筛选直接下载数据

    docs = list(mpr.materials.summary.search(
        elements=[NA_ELEMENT],
        band_gap=(BAND_GAP_MIN, None),
        fields=fields,
    ))

total_hits = len(docs)
targets = docs[:LIMIT] if (LIMIT and LIMIT > 0) else docs

print(f"命中材料数: {total_hits}")
if LIMIT and 0 < LIMIT < total_hits:
    print(f"按 LIMIT={LIMIT} 截取，本次处理: {len(targets)}")

def _sym_attr(sym, name):
    """兼容 symmetry 为 dict 或 SymmetryData 对象两种情况。"""
    if sym is None:
        return None
    if isinstance(sym, dict):
        return sym.get(name)
    return getattr(sym, name, None)


records = []
for d in targets:
    records.append({
        # 用 str() 把 MPID 对象转成普通字符串，否则 sort_values 会触发
        # MPID 的严格比较，遇到非 prefix-number / ULID 格式的 id 就报 ValueError
        "material_id": str(d.material_id),
        "formula_pretty": d.formula_pretty,
        "formula_anonymous": d.formula_anonymous,
        "band_gap": d.band_gap,
        "energy_above_hull": d.energy_above_hull,
        "density": d.density,
        "volume": getattr(d, "volume", None),
        "nsites": d.nsites,
        "nelements": d.nelements,
        "elements": ",".join(str(e) for e in d.elements) if d.elements else "",
        "crystal_system": _sym_attr(d.symmetry, "crystal_system"),
        "spacegroup_symbol": _sym_attr(d.symmetry, "symbol"),
        "spacegroup_number": _sym_attr(d.symmetry, "number"),
    })

df = pd.DataFrame(records).sort_values("material_id").reset_index(drop=True)
df.to_csv(SUMMARY_CSV, index=False)
df.to_json(SUMMARY_JSON, orient="records", indent=2, force_ascii=False)

print(f"信息表已保存: {SUMMARY_CSV}  ({len(df)} 行)")
print(f"band_gap 范围: {df['band_gap'].min():.3f} ~ {df['band_gap'].max():.3f} eV")

# ---- 写 CIF（带进度条）----
# 有 tqdm 用 tqdm，没有则退回每 200 条打印一次
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except ImportError:
    _HAS_TQDM = False
    print("未安装 tqdm（pip install tqdm），退回到每 200 条打印一次。")

report_rows = []
ok = skipped = errors = 0
total = len(targets)

iterator = tqdm(targets, total=total, desc="写出 CIF", unit="cif") if _HAS_TQDM else targets

for i, d in enumerate(iterator, start=1):
    mid = str(d.material_id)
    target = OUTDIR / f"{mid}.cif"
    row = {"material_id": mid, "path": str(target), "status": "", "error": ""}

    if SKIP_EXISTING and target.exists():
        row["status"] = "skipped"
        skipped += 1
    else:
        try:
            CifWriter(d.structure).write_file(str(target))
            row["status"] = "ok"
            ok += 1
        except Exception as e:
            row["status"] = "error"
            row["error"] = f"{type(e).__name__}: {e}"
            errors += 1

    report_rows.append(row)

    if _HAS_TQDM:
        iterator.set_postfix(ok=ok, skip=skipped, err=errors)  # 实时计数
    elif i % 200 == 0 or i == total:
        print(f"[{i}/{total}] ok={ok}  skipped={skipped}  errors={errors}")

report = pd.DataFrame(report_rows)
report.to_csv(REPORT_CSV, index=False)

# 把下载状态合并进信息表，得到最终清单
final = df.merge(report[["material_id", "status", "error"]], on="material_id", how="left")
final.to_csv(FINAL_CSV, index=False)

print("\n===== 汇总 =====")
print(f"命中: {total_hits} 条；本次处理: {len(df)} 条")
print(f"CIF 目录: {OUTDIR.resolve()}")
print(f"  写出 ok={ok}  跳过 skipped={skipped}  失败 errors={errors}")
print(f"信息表:   {SUMMARY_CSV.name}")
print(f"下载报告: {REPORT_CSV.name}")
print(f"最终清单: {FINAL_CSV.name}")
print("\n状态分布:")
print(final["status"].value_counts())

Retrieving SummaryDoc documents: 100%|██████████| 8253/8253 [00:34<00:00, 236.67it/s]


命中材料数: 8253
信息表已保存: cif/na_bandgap_ge1/summary.csv  (8253 行)
band_gap 范围: 1.001 ~ 8.592 eV


写出 CIF: 100%|██████████| 8253/8253 [00:10<00:00, 768.64cif/s, err=0, ok=8243, skip=10] 



===== 汇总 =====
命中: 8253 条；本次处理: 8253 条
CIF 目录: /Users/north./Documents/JXNU/code/self_study/21-mp-download/cif/na_bandgap_ge1
  写出 ok=8243  跳过 skipped=10  失败 errors=0
信息表:   summary.csv
下载报告: download_report.csv
最终清单: summary_with_status.csv

状态分布:
status
ok         8243
skipped      10
Name: count, dtype: int64
